In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, privacy

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
cfg = common.load_config(dbutils.widgets.get("env"))
g = f"{cfg['catalog']}.gold"
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print("PASS" if ok else "FAIL", name, detail)

def one(sql):
    return spark.sql(sql).first()[0]

In [0]:
dims = {"dim_hospital": ("hospital_key", "hospital_id", 6, None),
        "dim_doctor": ("doctor_key", "doctor_id", 74, 61),
        "dim_patient": ("patient_key", "patient_id", 655, 500)}

for t, (k, nid, rows, current) in dims.items():
    s = spark.sql(f"""SELECT count(*) n, count(DISTINCT {k}) keys, sum(CASE WHEN {k} = -1 THEN 1 ELSE 0 END) unknown
                      FROM {g}.{t}""").first()
    check(f"{t}: {rows} rows, unique keys, one unknown", (s["n"], s["keys"], s["unknown"]) == (rows, rows, 1), tuple(s))
    if current:
        cur = one(f"SELECT count(*) FROM {g}.{t} WHERE is_current")
        many = one(f"SELECT count(*) FROM (SELECT {nid} FROM {g}.{t} WHERE is_current GROUP BY {nid} HAVING count(*) > 1)")
        check(f"{t}: one current version per id ({current})", cur == current and many == 0, (cur, many))
        bad = one(f"""SELECT count(*) FROM (SELECT valid_to, lead(valid_from) OVER (PARTITION BY {nid} ORDER BY valid_from) nxt,
                      row_number() OVER (PARTITION BY {nid} ORDER BY valid_from) rn, valid_from FROM {g}.{t})
                      WHERE (nxt IS NOT NULL AND valid_to <> nxt) OR (rn = 1 AND valid_from <> TIMESTAMP'1900-01-01')""")
        check(f"{t}: versions join up with no gap or overlap", bad == 0, bad)

personal = {"full_name", "phone", "aadhaar", "dob"}
leak = [(t, c) for t in dims for c in spark.table(f"{g}.{t}").columns if c in personal]
check("no name, phone, national id or date of birth in Gold", not leak, str(leak))
check("dim_patient.id_fingerprint is masked", privacy.masks_now(spark, cfg["catalog"]).get(("gold", "dim_patient", "id_fingerprint")) == "mask_token")

print(f"\nt18_gold_dims_bhg: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t18_gold_dims_bhg failed")